# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [ ]:
import os
from dotenv import load_dotenv


load_dotenv() # loads .env into the environment

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set")

In [3]:
from openai import OpenAI

#Function to call the model.
def return_OAI(user_message):
    client = OpenAI()

    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [23]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {'tables': ['employees', 'salary', 'studies'],
        'definition': ['employees: employee_id (PK), name', 'salary: employee_id (FK), year, amount', 'studies: employee_id (FK), degree_level']
}

df = pd.DataFrame(data)
df

,tables,definition
0,employees,"employees: employee_id (PK), name"
1,salary,"salary: employee_id (FK), year, amount"
2,studies,"studies: employee_id (FK), degree_level"


In [24]:
text_tables = '\n'.join([f"{row['tables']}: {row['definition']}" for index, row in df.iterrows()])

In [25]:
print(text_tables)

employees: employees: employee_id (PK), name
salary: salary: employee_id (FK), year, amount
studies: studies: employee_id (FK), degree_level


In [36]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return only the minimal set of tables strictly required to compute the result. Do not include tables used only for semantic completeness.
Return the table names in a json format.
###User Question:
{question}
"""


In [27]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question="Which employee has the highest salary?")

In [22]:
print(return_OAI(pqt1))

```json
{
    "tables": ["employees", "salary"]
}
```


In [28]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question="What is the education institution with most employees?")

In [29]:
print(return_OAI(pqt3))

```json
{
    "tables": ["employees", "studies"]
}
```


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

### Stress testing our table definition

In [32]:
# Level 1
p_easy_1 = prompt_question_tables.format(
            tables=text_tables,
            question="What is the average salary in 2022?"
)

p_easy_2 = prompt_question_tables.format(
            tables=text_tables,
            question="List all employees?"
)

p_easy_3 = prompt_question_tables.format(
            tables=text_tables,
            question="How many employees have a master's degree?"
)

print(
    return_OAI(p_easy_1),
    "\n",
    return_OAI(p_easy_2),
    "\n",
    return_OAI(p_easy_3),
)

```json
{
    "tables": ["salary"]
}
``` 
 ```json
{
    "tables": ["employees"]
}
``` 
 ```json
{
    "tables": ["employees", "studies"]
}
```


In [33]:
# Level 2
p_medium_1 = prompt_question_tables.format(
    tables=text_tables,
    question="List employee name who earned more than 100k in 2021."
)
p_medium_2 = prompt_question_tables.format(
    tables=text_tables,
    question="How many employees have a PhD?"
)
p_medium_3 = prompt_question_tables.format(
    tables=text_tables,
    question="Show employees and their salary in 2023."
)

print(
    return_OAI(p_medium_1),
    "\n",
    return_OAI(p_medium_2),
    "\n",
    return_OAI(p_medium_3),
)

```json
{
    "tables": ["employees", "salary"]
}
``` 
 ```json
{
    "tables": ["employees", "studies"]
}
``` 
 ```json
{
    "tables": ["employees", "salary"]
}
```


In [37]:
# Level 3
p_hard_1 = prompt_question_tables.format(
    tables=text_tables,
    question="Employees with a master's degree earning more than 90k in 2022."
)
p_hard_2 = prompt_question_tables.format(
    tables=text_tables,
    question="Count people who studied and have salary records in 2021"
)
p_hard_3 = prompt_question_tables.format(
    tables=text_tables,
    question="Average salary of people with a bachelor's degree."
)

print(
    return_OAI(p_hard_1),
    "\n",
    return_OAI(p_hard_2),
    "\n",
    return_OAI(p_hard_3),
)


```json
{
    "tables": ["employees", "salary", "studies"]
}
``` 
 {
    "tables": ["employees", "salary", "studies"]
} 
 ```json
["salary", "studies"]
```


In [38]:
# Level 4 - The stress test
p_stress_1 = prompt_question_tables.format(
    tables=text_tables,
    question="Total payroll cost for PhD degree holders in 2023."
)
p_stress_2 = prompt_question_tables.format(
    tables=text_tables,
    question="People who have salary records but no education records"
)
p_stress_3 = prompt_question_tables.format(
    tables=text_tables,
    question="Collaborators hired recently with no salary in 2022."
)

print(
    return_OAI(p_stress_1),
    "\n",
    return_OAI(p_stress_2),
    "\n",
    return_OAI(p_stress_3),
)

```json
{
    "tables": ["salary", "studies"]
}
``` 
 ```json
["employees", "salary"]
``` 
 ```json
{
    "tables": ["employees", "salary"]
}
```


### Summary

Starting from vague table descriptions, routing was inconsistent because the schema lacked clear ownership and relational signals. Once we shifted to compact, column-level definitions with explicit join keys and routing-relevant fields (e.g., filter and aggregation columns), table selection became reliably driven by lexical alignment and shared identifiers. The model performed well on positive, filter-based and multi-table queries, correctly inferring joins through consistent key naming. Adding an explicit minimality constraint reduced over-selection caused by semantic entity framing (e.g., “people” or “employees”). The main limitation surfaced with negation and absence logic (e.g., “no education records”), where routing degraded despite clear schema signals. Overall, performance scaled with structural clarity, and the practical ceiling appeared in handling anti-join semantics and linguistic bias rather than relational complexity itself.